In [ ]:
from __future__ import annotations
from geopy.geocoders import Nominatim
from dataclasses import dataclass
import xarray as xr
import pandas as pd
import numpy as np

@dataclass
class Location:
    name: str
    latitude: float
    longitude: float

_geolocator = None

def geocode_city(city: str, user_agent: str = "pogoda-app") -> Location:
    """Geocode a city name into coordinates using Nominatim.

    Parameters
    ----------
    city: str
        City name (can include country, e.g., "Szczecin, Poland").
    user_agent: str
        Required by Nominatim usage policy.

    Returns
    -------
    Location

    Raises
    ------
    ValueError: if no result is found.
    """
    global _geolocator
    if _geolocator is None:
        _geolocator = Nominatim(user_agent=user_agent, timeout=10)
    result = _geolocator.geocode(city)
    if not result:
        raise ValueError(f"City not found: {city}")
    return Location(name=result.address, latitude=result.latitude, longitude=result.longitude)


In [ ]:

#rain = xr.open_dataset("rr_ens_mean_0.1deg_reg_v31.0e.nc")

In [ ]:
ds = xr.open_dataset("tg_ens_mean_0.25deg_reg_2011-2024_v31.0e.nc")
df = ds["tg"].resample(time="1MS").mean(dim=("time"))

df

In [ ]:
# Drop months that are entirely NaN
df = df.dropna(dim="time", how="all")

# Drop lat/lon points that are entirely NaN across all months
df = df.dropna(dim="latitude", how="all")
df = df.dropna(dim="longitude", how="all")


In [ ]:
df

In [ ]:
import xarray as xr
import numpy as np

ds = xr.open_dataset("rr_ens_mean_0.1deg_reg_v31.0e.nc")

df = (
    ds["rr"]
      .resample(time="1MS").sum(dim="time")
      .to_dataframe(name="rr")
      .dropna()               # removes any (time, lat, lon) rows that are NaN
      .reset_index()
)
df.to_parquet("rr_monthly_all.parquet", engine="pyarrow", compression="snappy", index=False)

In [ ]:
print(df)


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

def monthly_aggregate_to_parquet(
    da: xr.DataArray,
    agg: str = "mean",                 # "mean" for TG, "sum" for RR
    out_path: str | Path = "monthly.parquet",
    chunks: dict | None = None         # e.g., {"time": 180, "latitude": 200, "longitude": 200}
) -> pd.DataFrame:
    """
    Aggregate a daily gridded DataArray (time, lat, lon) to monthly per-cell stats and save to Parquet.

    Output columns: latitude, longitude, year, month, <varname>, n
      - <varname> is da.name (e.g., "tg" or "rr")
      - n is the number of non-NaN daily rows contributing to that month
    """
    if chunks:
        da = da.chunk(chunks)

    if da.name is None:
        # keep a sane value column name if DataArray is unnamed
        da = da.rename("value")

    res = da.resample(time="1MS")
    if agg == "mean":
        agg_da = res.mean(dim="time", skipna=True)
    elif agg == "sum":
        agg_da = res.sum(dim="time", skipna=True)
    else:
        raise ValueError("agg must be 'mean' or 'sum'")

    # count of non-NaN daily values per month
    n_da = res.count(dim="time")

    # combine; drop months with zero valid days
    ds_month = xr.Dataset({da.name: agg_da, "n": n_da})
    ds_month = ds_month.where(ds_month["n"] > 0, drop=True)

    # -> tidy pandas
    df = ds_month.to_dataframe().reset_index()
    df["year"] = df["time"].dt.year.astype("int16")
    df["month"] = df["time"].dt.month.astype("int8")
    df = df.drop(columns=["time"])

    # reorder columns
    df = df[["latitude", "longitude", "year", "month", da.name, "n"]]

    # save parquet
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_path, index=False)

    return df

In [ ]:
ds = xr.open_dataset("tg_ens_mean_0.25deg_reg_v31.0e.nc")#, chunks={"time": 180})

In [ ]:
df_tg = monthly_aggregate_to_parquet(ds["tg"], agg="mean", out_path="tg_monthly.parquet")

In [ ]:
df_tg